# 04 Modeling\n\nGoal: test whether sentiment features help predict next-day S&P 500 direction using a time-based split.

In [ ]:
from pathlib import Path\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\nDATA_PATH = PROJECT_ROOT / "data" / "processed" / "daily_sentiment_macro.csv"

In [ ]:
daily = pd.read_csv(DATA_PATH, parse_dates=["Date"])\nfeatures = [\n    "sentiment_compound",\n    "sentiment_pos",\n    "sentiment_neg",\n    "headline_count",\n    "vix",\n    "treasury_10y",\n    "term_spread_10y_2y",\n    "fed_funds_rate",\n    "cpi_yoy",\n    "unemployment_rate",\n]\ntarget = "direction_next_day"\n\nsplit_idx = int(len(daily) * 0.8)\ntrain = daily.iloc[:split_idx].copy()\ntest = daily.iloc[split_idx:].copy()\n\nmodel = Pipeline([\n    ("scaler", StandardScaler()),\n    ("logistic", LogisticRegression(max_iter=1000)),\n])\nmodel.fit(train[features], train[target])\n\npred = model.predict(test[features])\npred_prob = model.predict_proba(test[features])[:, 1]\n\nprint(f"Train: {train['Date'].min().date()} to {train['Date'].max().date()}")\nprint(f"Test: {test['Date'].min().date()} to {test['Date'].max().date()}")\nprint(f"Accuracy: {accuracy_score(test[target], pred):.3f}")\nprint(f"ROC AUC: {roc_auc_score(test[target], pred_prob):.3f}")\nprint(classification_report(test[target], pred))\nprint(confusion_matrix(test[target], pred))